# ML-1 interest model
Train the explainable baseline and final random forest from the synthetic participant data.

In [1]:
from datetime import date
import json
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
DATA_PATH = Path('../data/ml1_participants.csv')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
raw_features = [
    'age', 'sex', 'employment', 'education', 'region', 'smoker', 'bmi',
    'activity_level', 'flag_diabetes', 'flag_hypertension',
    'flag_high_cholesterol', 'flag_respiratory', 'flag_mental', 'flag_sleep',
    'flag_skin', 'motivation', 'availability', 'max_travel_minutes',
]
data = pd.read_csv(DATA_PATH)
categorical = ['sex', 'employment', 'education', 'region', 'activity_level', 'motivation', 'availability']
features = pd.get_dummies(data[raw_features], columns=categorical)
feature_columns = list(features.columns)
X_train, X_test, y_train, y_test = train_test_split(
    features, data['label'], test_size=0.3, stratify=data['label'], random_state=42
)
print(len(X_train), len(X_test), len(feature_columns))


2100 900 39


In [2]:
baseline = DecisionTreeClassifier(max_depth=5, random_state=42)
baseline.fit(X_train, y_train)
model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
model.fit(X_train, y_train)
train_accuracy = accuracy_score(y_train, model.predict(X_train))
test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
baseline_tree_accuracy = accuracy_score(y_test, baseline.predict(X_test))
macro_f1 = f1_score(y_test, test_predictions, average="macro")
chance_accuracy = 1 / 9
print('Train accuracy:', train_accuracy)
print('Test accuracy:', test_accuracy)
print('Baseline tree test accuracy:', baseline_tree_accuracy)
print('Chance:', chance_accuracy)
print('Macro F1:', macro_f1)
print('Confusion matrix:\n', confusion_matrix(y_test, test_predictions, labels=model.classes_))


Train accuracy: 0.7671428571428571
Test accuracy: 0.5933333333333334
Baseline tree test accuracy: 0.5044444444444445
Chance: 0.1111111111111111
Macro F1: 0.5467124038241425
Confusion matrix:
 [[ 41   8  16  12   5   8   3   4   0]
 [  7 124   3   2   2   4   0   0   3]
 [  7   3 113   3   4  14   1   4   0]
 [  4  25  12  36   1   9   3   7   5]
 [  0   8   4   2  50   3   0   2   0]
 [  7   7  15  12   1 105   1   5   1]
 [  4   5   4   5   2   2  16   3   0]
 [  7   2   3   3   5   7   1  30   2]
 [  3  26  11   9   7   3   1   4  19]]


In [3]:
import joblib
joblib.dump(model, MODEL_DIR / 'ml1_interest.joblib')
metadata = {
    'model_name': 'ml1',
    'version': 'ml1-v1.1',
    'trained_at': date.today().isoformat(),
    'algorithm': 'RandomForestClassifier',
    'params': model.get_params(),
    'raw_features': raw_features,
    'feature_columns': feature_columns,
    'classes': list(model.classes_),
    'metrics': {
        'train_accuracy': float(train_accuracy),
        'test_accuracy': float(test_accuracy),
        'baseline_tree_accuracy': float(baseline_tree_accuracy),
        'chance_accuracy': float(chance_accuracy),
        'macro_f1': float(macro_f1),
    },
    'n_train': len(X_train),
    'n_test': len(X_test),
}
with (MODEL_DIR / 'ml1_interest_meta.json').open('w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)
